# 🥈 Silver — clean the data

Reads `my.bronze.sales` and fixes what's wrong with it:

| Problem in the file | Fix |
|---|---|
| the same row twice | drop exact duplicates, then keep one row per `order_id` |
| text columns | proper types — numbers, decimals, dates |
| two date formats (`2026-03-05`, `05/03/2026`) | try both |
| `"  finland"`, `FINLAND`, `coffee ` | trim spaces, capitalise |
| empty city | `Unknown` |
| no order id / date / quantity / price, `N/A` price, quantity ≤ 0 | not usable → `my.silver.sales_rejected` |

Good rows go to `my.silver.sales`, with `revenue = quantity × unit_price`.

In [ ]:
from pyspark.sql import functions as F

bronze = spark.table("my.bronze.sales")
steps = [("rows in bronze", bronze.count())]

## 1 · Duplicates

In [ ]:
df = bronze.drop("_source_file", "_ingested_at").dropDuplicates()     # identical rows → one
steps.append(("after dropping exact duplicates", df.count()))

## 2 · Types and tidy values
`try_cast` / `try_to_timestamp` give **NULL** instead of an error when a value doesn't fit (`N/A` as
a price, a date in the other format) — the bad values are then easy to find.

In [ ]:
df = (df
      .withColumn("order_id",   F.expr("try_cast(order_id AS int)"))
      .withColumn("quantity",   F.expr("try_cast(quantity AS int)"))
      .withColumn("unit_price", F.expr("try_cast(unit_price AS decimal(10,2))"))     # 'N/A' → NULL
      .withColumn("order_date", F.coalesce(F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
                                           F.expr("try_to_timestamp(order_date, 'dd/MM/yyyy')"))
                                 .cast("date"))
      .withColumn("country",  F.initcap(F.trim("country")))      # '  FINLAND ' → 'Finland'
      .withColumn("category", F.initcap(F.trim("category")))
      .withColumn("city",     F.coalesce(F.trim("city"), F.lit("Unknown"))))
df.groupBy("country").count().orderBy("country").show()

## 3 · Nulls and bad values → rejected

In [ ]:
valid = F.expr("""order_id IS NOT NULL AND order_date IS NOT NULL
                  AND quantity IS NOT NULL AND quantity > 0 AND unit_price IS NOT NULL""")
rejected = df.filter(~valid)
good = df.filter(valid)
steps.append(("rejected (missing / bad values)", rejected.count()))

good = good.dropDuplicates(["order_id"])                               # one row per order
steps.append(("after one row per order_id", good.count()))

silver = good.withColumn("revenue", (F.col("quantity") * F.col("unit_price")).cast("decimal(12,2)"))

for step, rows in steps:
    print(f"{step:<34} {rows:>6}")

## 4 · Write Silver

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS my.silver")
silver.writeTo("my.silver.sales").using("iceberg").createOrReplace()
rejected.writeTo("my.silver.sales_rejected").using("iceberg").createOrReplace()
print("my.silver.sales:", spark.table("my.silver.sales").count(), "rows")
print("my.silver.sales_rejected:", spark.table("my.silver.sales_rejected").count(), "rows")

In [ ]:
%%sql
SELECT * FROM my.silver.sales_rejected LIMIT 10